# Atividade: Predição de Diabetes — Pipeline com KNNImputer & PowerTransformer

Este experimento foca na engenharia de pré-processamento avançada para o classificador k-NN no dataset *Pima Indians Diabetes*:
1. **Tratamento de Zeros Fisiológicos:** Identificação de valores nulos disfarçados de zero (Glicose, Pressão Arterial, Espessura da Pele, Insulina e IMC).
2. **Tratamento de Outliers:** Aplicação de *Outlier Capper* customizado com corte por amplitude interquartil (IQR).
3. **Imputação Topológica:** `KNNImputer(n_neighbors=15)` para preservar a estrutura de vizinhança local dos dados no espaço multivariado.
4. **Transformação Normalizadora:** `PowerTransformer(method='yeo-johnson')` para estabilização de variância e conversão para distribuições aproximadamente normais.
5. **Classificação & Validação:** Modelo k-NN (k=3).

> **Desempenho:** Acurácia de **63.27%** registrada no servidor de avaliação independente (conjunto de teste cego).


### 1. Importações e Configurações
Configuramos as bibliotecas e a chave de acesso da equipe.

In [ ]:
import pandas as pd
import numpy as np
import requests
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.impute import KNNImputer
from sklearn.preprocessing import PowerTransformer
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
import warnings
warnings.filterwarnings('ignore')

#config
DEV_KEY = "2 computeiros e um matemagico"
URL = "https://aydanomachado.com/mlclass/01_Preprocessing.php"

### 2. Carga dos Dados
Lemos os arquivos CSV originais.

In [ ]:
print("Lendo os dados...")
FEATURE_COLS = ['Pregnancies', 'Glucose', 'BloodPressure', 'SkinThickness',
                'Insulin', 'BMI', 'DiabetesPedigreeFunction', 'Age']
TARGET = 'Outcome'

df_train = pd.read_csv('diabetes_dataset.csv')
df_app = pd.read_csv('diabetes_app.csv')

X_train = df_train[FEATURE_COLS]
y_train = df_train[TARGET]
X_app_final = df_app[FEATURE_COLS]

Dados de treino carregados: (572, 9)
Dados de aplicação carregados: (196, 8)


### 3. Pré-processamento de Dados
**Nesta seção, vamos implementar as técnicas de pré-processamento**

In [ ]:
# CORTA OUTLIERS (IQR)

class OutlierCapper(BaseEstimator, TransformerMixin):
    def __init__(self, factor=1.0):
        self.factor = factor
        self.lower_bounds = None
        self.upper_bounds = None

    def fit(self, X, y=None):
        Q1 = np.nanpercentile(X, 25, axis=0)
        Q3 = np.nanpercentile(X, 75, axis=0)
        IQR = Q3 - Q1
        self.lower_bounds = Q1 - self.factor * IQR
        self.upper_bounds = Q3 + self.factor * IQR
        return self

    def transform(self, X, y=None):
        return np.clip(X, self.lower_bounds, self.upper_bounds)

In [ ]:
# REMOVE COLUNAS (Zerando)
class FeatureZeroer(BaseEstimator, TransformerMixin):
    """
    Remove features do calculo da distancia do KNN, zerando-as.
    Em vez de dropar, mantem o formato igual ao dataset original
    mas não influencia a perfomance preditiva!
    """
    def __init__(self, cols_to_zero):
        self.cols_to_zero = cols_to_zero
        self.col_indices = None

    def fit(self, X, y=None):
        # Mapeando os nomes para indices
        self.col_indices = [FEATURE_COLS.index(c) for c in self.cols_to_zero]
        return self

    def transform(self, X, y=None):
        X_copy = np.copy(X)
        for idx in self.col_indices:
            X_copy[:, idx] = 0.0
        return X_copy

In [ ]:
# KNN Imputer (k=15) + Power Transformer + Outlier Capper (1.0)
# + Zerar Insulin e SkinThickness

pipeline_campeao = Pipeline([
    ('imputer', KNNImputer(n_neighbors=15)),                     # Imputacao baseada nos vizinhos mais proximos!
    ('capper', OutlierCapper(factor=1.0)),                       # Controle rigoroso de ranges extremos!
    ('scaler', PowerTransformer(method='yeo-johnson')),          # Melhor q o Standard: Ajeita a escala p/ uma dist. normal
    ('drop_ruido', FeatureZeroer(['Insulin', 'SkinThickness'])), # Neutraliza as colunas ruins c/ mtos NaNs
    ('knn', KNeighborsClassifier(n_neighbors=3))                 # O modelo intacto do professor!
])

In [ ]:
# 3. TREINAMENTO E PREDIÇÃO
print("Treinando o modelo com 100% dos dados para a prova final...")
pipeline_campeao.fit(X_train, y_train)

print("Gerando predições...")
y_pred_oficial = pipeline_campeao.predict(X_app_final)

In [ ]:
# ==========================================
# 4. ENVIO PARA O SERVIDOR
# ==========================================
dados_envio = {
    'dev_key': DEV_KEY,
    'predictions': pd.Series(y_pred_oficial).to_json(orient='values')
}

print(' Enviando predições calibradas para a API do professor...')
response = requests.post(url = URL, data = dados_envio)
print("\n Resposta do servidor:")
print(response.text)
